# Notebook 04 — PCA for Single-Cell Data

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 04 of 12  
**Time estimate:** 80 minutes

> The cell × gene matrix has thousands of dimensions.
> PCA compresses it to ~50 principal components that capture most of the biological variance.
> Everything downstream — UMAP, clustering, DE — operates in PCA space, not gene space.

---
## Step 1 — Motivation

After HVG selection we have ~2000 genes × N cells. Computing k-nearest neighbours in
2000-dimensional space is slow and suffers from the curse of dimensionality. PCA
reduces to ~50 components that capture most structure while discarding noise.

---
## Step 2 — Intuition

Imagine plotting cells in 2D where axes are Gene1 and Gene2. If genes are correlated
(T-cell marker genes all high in T cells), the data lies along a diagonal — one
direction captures most variance. PCA finds these directions (principal components)
by rotating axes to align with maximum variance.

**In scRNA-seq:**  
PC1 often separates the largest cell-type populations. PC2 separates the next-largest
split. The first 50 PCs collectively capture the major biological structure;
remaining PCs are dominated by technical noise.

---
## Step 3 — Biological Background

**Why PCA works for scRNA-seq:**  
Co-expressed genes (e.g. all T-cell activation genes) move together across cells.
PCA captures these co-expression programs as principal components — each PC represents
a biological program or technical source of variation (e.g. cell-cycle, library quality).

**Cell-cycle effect:**  
A major confound in scRNA-seq is cell cycle — cells in G1/S/G2 express different sets
of genes regardless of cell type. Often PC1 or PC2 captures cell-cycle variation.
Regression of cell-cycle scores is sometimes applied before PCA.

**Number of PCs to use:**  
The elbow plot (variance explained vs. PC number) shows a kink where biological signal
transitions to noise. Typically 10–50 PCs are used for kNN graph construction. Too few
loses biological signal; too many adds noise and slows downstream steps.

---
## Step 4 — Mathematical Explanation

**Centering:**  
$$X_{\text{centered}} = X - \bar{X}$$
where $\bar{X}$ is the gene-wise mean (subtract mean of each column).

**SVD:**  
$$X_{\text{centered}} = U \Sigma V^T$$

where $U \in \mathbb{R}^{n \times k}$ (cell coordinates), $\Sigma \in \mathbb{R}^{k \times k}$
(singular values, descending), $V \in \mathbb{R}^{p \times k}$ (gene loadings).

**Principal components (PC scores):**  
$$Z = U \Sigma = X_{\text{centered}} V$$
$Z_{ik}$ is the coordinate of cell $i$ along PC $k$.

**Variance explained:**  
$$\text{var\_explained}_k = \frac{\sigma_k^2}{\sum_j \sigma_j^2}$$

where $\sigma_k$ is the $k$-th singular value. The fraction of total variance explained
by the first $K$ components is $\sum_{k=1}^K \sigma_k^2 / \sum_j \sigma_j^2$.

In [ ]:
# Step 6 — Python: PCA from scratch with SVD on single-cell data
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# ---- Simulate normalized + HVG-selected data ----
N_CELLS = 300
N_HVG = 400  # post-HVG selection
cell_types = rng.choice(['T_cell', 'B_cell', 'Monocyte'], N_CELLS, p=[0.4, 0.35, 0.25])

def make_structured_data(n_cells, n_genes, cell_types, rng):
    """Simulate log-normalized data with three cell-type clusters."""
    # Each cell type has its own expression program
    programs = {
        'T_cell':   {'idx': range(0, 80),   'level': 2.5},
        'B_cell':   {'idx': range(80, 160),  'level': 2.0},
        'Monocyte': {'idx': range(160, 260), 'level': 3.0},
    }
    X = rng.normal(0, 0.3, (n_cells, n_genes))  # background noise
    for i, ct in enumerate(cell_types):
        p = programs[ct]
        X[i, list(p['idx'])] += p['level'] + rng.normal(0, 0.4, len(list(p['idx'])))
    return np.clip(X, 0, None)

X_hvg = make_structured_data(N_CELLS, N_HVG, cell_types, rng)

# ---- PCA from scratch ----
def run_pca(X, n_components=50):
    """PCA via full SVD (use truncated SVD for large data)."""
    # 1. Center (subtract gene-wise mean)
    X_centered = X - X.mean(axis=0)

    # 2. SVD (economy/thin SVD — only n_components columns returned)
    # np.linalg.svd full_matrices=False gives economy SVD
    U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)

    # 3. Take top n_components
    U   = U[:, :n_components]
    s   = s[:n_components]
    Vt  = Vt[:n_components, :]

    # 4. PC scores (cell coordinates)
    Z = U * s  # same as U @ np.diag(s)

    # 5. Variance explained
    # Full variance = sum of ALL squared singular values
    _, s_full, _ = np.linalg.svd(X_centered, full_matrices=False)
    var_explained = s**2 / (s_full**2).sum()

    return {
        'Z': Z,           # (n_cells, n_components) — PC scores
        'Vt': Vt,         # (n_components, n_genes) — gene loadings
        's': s,           # singular values
        'var_explained': var_explained,
    }

pca = run_pca(X_hvg, n_components=50)
Z = pca['Z']

print('PCA complete')
print(f'  PC scores shape: {Z.shape}')
print(f'  PC1 variance explained: {pca["var_explained"][0]*100:.1f}%')
print(f'  PC2 variance explained: {pca["var_explained"][1]*100:.1f}%')
print(f'  Top 10 PCs total: {pca["var_explained"][:10].sum()*100:.1f}%')
print(f'  Top 50 PCs total: {pca["var_explained"].sum()*100:.1f}%')

In [ ]:
# Step 7 — Visualization: Elbow plot and PC1 vs PC2 scatter
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ct_colors = {'T_cell': 'steelblue', 'B_cell': 'tomato', 'Monocyte': 'forestgreen'}
colors = [ct_colors[ct] for ct in cell_types]

# Panel A: Elbow plot
ax = axes[0]
var_pct = pca['var_explained'] * 100
ax.plot(range(1, 51), var_pct, 'o-', lw=1.5, ms=3, color='steelblue')
ax.axvline(15, color='red', ls='--', alpha=0.7, label='typical cutoff (15 PCs)')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('A. Elbow plot — variance per PC')
ax.legend(fontsize=8)

# Panel B: Cumulative variance
ax = axes[1]
ax.plot(range(1, 51), np.cumsum(var_pct), 'o-', lw=1.5, ms=3, color='tomato')
ax.axhline(80, color='grey', ls='--', label='80% variance')
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative Variance (%)')
ax.set_title('B. Cumulative variance explained')
ax.legend(fontsize=8)

# Panel C: PC1 vs PC2 scatter
ax = axes[2]
for ct, col in ct_colors.items():
    mask = np.array(cell_types) == ct
    ax.scatter(Z[mask, 0], Z[mask, 1], c=col, s=12, alpha=0.7, label=ct)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('C. PC1 vs. PC2 — cell types separate')
ax.legend(fontsize=8)

plt.suptitle('Module 10 NB04: PCA for Single-Cell Data', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scrna_pca.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement `run_pca()` from scratch using only `np.linalg.svd`. Do NOT import
   sklearn or scanpy for this exercise.
2. From the elbow plot, identify the number of PCs where the curve visibly flattens.
3. What do the gene loadings `Vt[0, :]` tell you about PC1? Which gene has the highest
   loading on PC1?
4. Run PCA with `n_components=2` vs `n_components=50`. Does the PC1 vs. PC2 scatter
   change? Why or why not?

---
## Step 10 — Quiz

1. What does the SVD decomposition $X = U \Sigma V^T$ give you? What is the shape
   of each matrix for a 300-cell × 400-gene input?
2. Why must data be centered before PCA?
3. What is the elbow plot and how do you use it to choose `n_components`?
4. In scanpy, PCA is run with `sc.tl.pca(adata, n_comps=50)`. Where are the results
   stored in the AnnData object?
5. What does it mean biologically that PC1 explains 30% of variance?

---
## Step 12 — Reflection

> *[In one paragraph: explain PCA to someone who knows linear algebra but has
> never seen single-cell data. What is the biological interpretation of the
> first few principal components?]*

---
*Next: `05_umap_tsne_visualization.ipynb`*